# 04 · Result analysis
Charts and summary for the baseline. 30 samples demonstrate the pipeline; they are **not** statistically conclusive,
and no compression method has been run, so there are no memory-reduction or quality-retention numbers yet.

In [ ]:
# Run from anywhere: find the project root and make it importable
import os, sys
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "main.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path[:0] = [str(ROOT), str(ROOT / "scripts")]
PY = sys.executable
print("project root:", ROOT)

In [ ]:
EXPERIMENT = "baseline_001"
from analyze_results import analyze
summary, charts = analyze(EXPERIMENT)

In [ ]:
from IPython.display import Image, display
for c in charts:
    display(Image(filename=str(c)))

In [ ]:
import pandas as pd
pd.read_csv(f"results/csv/{EXPERIMENT}_by_workflow.csv")

In [ ]:
# Long-context run, if you did it in notebook 03
import os
if os.path.exists("results/json/baseline_longctx_8k_d50.json"):
    s2, c2 = analyze("baseline_longctx_8k_d50")
    for c in c2[:2]:
        display(Image(filename=str(c)))

## Position sensitivity (if you ran section 4 of notebook 03)

In [ ]:
import glob
if glob.glob("results/json/position_full_d*.json"):
    from analyze_position import analyze as analyze_position
    table, chart = analyze_position("position_full")
    display(Image(filename=str(chart)))
    display(table)

## LongCodeU (if you ran section 5 of notebook 03)

In [ ]:
if os.path.exists("results/json/longcodeu_8k_full.json"):
    s3_, c3 = analyze("longcodeu_8k_full")
    for c in c3:
        if "longcodeu_by_task" in str(c):
            display(Image(filename=str(c)))
    display(pd.read_csv("results/csv/longcodeu_8k_full_longcodeu_by_task.csv"))

## Copy results to S3 (existing bucket only) and shut down
Set the bucket name, dry-run, then upload. Model weights are never uploaded.

In [ ]:
import os
os.environ["KVR_S3_BUCKET"] = ""          # <- your existing bucket name
os.environ["KVR_S3_PREFIX"] = "kv_cache_research"
if os.environ["KVR_S3_BUCKET"]:
    !{PY} scripts/sync_to_s3.py --dry-run
    # !{PY} scripts/sync_to_s3.py

### Shutdown checklist (GPU time is billed until the space is stopped)
1. Results saved: `results/json`, `results/csv`, `results/charts`, `results/server_report.json`, `requirements-lock.txt`.
2. Optional S3 copy done (cell above) or files downloaded from the file browser.
3. `git add results requirements-lock.txt && git commit && git push` (from a terminal).
4. Shut down all kernels (Running Terminals and Kernels panel).
5. **Stop the space**: SageMaker Studio → JupyterLab / Code Editor → your space → **Stop**. Closing the browser tab does not stop billing.
6. Check SageMaker → Inference → Endpoints and Training/Processing jobs show nothing running (this project never creates them).